# Dupin · Fase 3 — Régimen de evaluación honesto

Construye la vara de medir **antes** del modelo y produce el resultado central:
el **gap** entre el número optimista y el desplegable, descompuesto en sus dos
ejes.

**Decomposición 2×2** (reutiliza el mismo `evaluate()` para todo):

|                | Aleatorio (optimista) | Temporal (honesto) |
|----------------|-----------------------|--------------------|
| **Balance crudo** (fuga de etiqueta) | número fantasioso | — |
| **Features honestas**                | optimista          | **DESPLEGABLE**    |

- Eje A — **fuga de etiqueta**: balance crudo vs features honestas.
- Eje B — **fuga temporal**: split aleatorio vs temporal.

Punto de operación de cabecera: **recall @ revisar ≤1% de operaciones**.

## 1. Clonar el repo (mismo código de evaluación)

In [ ]:
from google.colab import userdata
import sys, subprocess
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
REPO = "alexxcode/dupin"
subprocess.run(["rm","-rf","/content/dupin"])
subprocess.run(["git","clone",f"https://{GITHUB_TOKEN}@github.com/{REPO}.git","/content/dupin"], check=True)
if "/content/dupin" not in sys.path: sys.path.insert(0,"/content/dupin")
subprocess.run(["git","-C","/content/dupin","log","--oneline","-1"])

In [ ]:
!pip -q install gcsfs pyarrow scikit-learn

## 2. Autenticación y carga de la matriz `feat-v1`

In [ ]:
from google.colab import auth
auth.authenticate_user()
import pandas as pd

PROJECT_ID  = "dupin-dupin"
BUCKET_FEAT = "dupin-dupin-features"
BUCKET_RAW  = "dupin-dupin-raw"
FEAT_URI = f"gs://{BUCKET_FEAT}/feat-v1/features.parquet"
RAW_URI  = f"gs://{BUCKET_RAW}/raw/paysim/PS_20174392719_1491204439457_log.csv"

matrix = pd.read_parquet(FEAT_URI, storage_options={"project": PROJECT_ID})
print("Matriz:", matrix.shape)

## 3. Viabilidad del split temporal

In [ ]:
from data.splits import DEFAULT_SPLIT, split_report
from evaluation import report as R

print(R.format_split({"split_report": split_report(matrix, DEFAULT_SPLIT)}))

## 4. Auditoría de fuga — features honestas (deben PASAR)

Ninguna feature honesta debería separar el fraude por sí sola (AUC ≈ 1).

In [ ]:
from evaluation.leakage_audit import audit
from features.config import FEATURE_NAMES

au = audit(matrix[FEATURE_NAMES], matrix["isFraud"], suspicious=0.98)
print("PASA (ninguna feature aislada filtra):", au["passed"])
print("\nTop AUC de feature individual:")
for k, v in list(au["single_feature_auc"].items())[:6]:
    print(f"  {k:20s} {v:.4f}")

## 5. Control positivo — la auditoría frente a las columnas de balance

Reconstruimos las columnas de balance (prohibidas) sobre la población de
superficie y verificamos que el aparato las distingue de las features honestas.

In [ ]:
bal = pd.read_csv(
    RAW_URI,
    usecols=["type","amount","oldbalanceOrg","newbalanceOrig","oldbalanceDest","newbalanceDest","isFraud","step"],
    storage_options={"project": PROJECT_ID},
)
bal = bal[bal["type"].isin(["TRANSFER","CASH_OUT"])].copy()
bal["errBalanceOrig"] = bal["newbalanceOrig"] + bal["amount"] - bal["oldbalanceOrg"]
bal["errBalanceDest"] = bal["oldbalanceDest"] + bal["amount"] - bal["newbalanceDest"]
BAL_COLS = ["oldbalanceOrg","newbalanceOrig","oldbalanceDest","newbalanceDest","errBalanceOrig","errBalanceDest"]

au_bal = audit(bal[BAL_COLS], bal["isFraud"], suspicious=0.98)
print("AUC individual de las columnas de balance (vs features honestas, todas <0.85):")
for k, v in au_bal["single_feature_auc"].items():
    print(f"  {k:20s} {v:.4f}")
print("\nNota: la fuga de etiqueta de PaySim es MULTI-columna; el número combinado")
print("se mide abajo con el modelo completo (eje A).")

## 6. Evaluación 2×2 — temporal vs aleatorio, honesto vs fuga

In [ ]:
from evaluation.evaluate import evaluate

BUDGET = 0.01

# Eje honesto: features de comportamiento causales.
rep_honest = evaluate(matrix, budget=BUDGET, return_scores=True)

# Eje fuga de etiqueta: mismas filas, columnas de balance crudas.
bal_matrix = bal[["step","isFraud"] + BAL_COLS].reset_index(drop=True)
rep_leaky = evaluate(bal_matrix, feature_names=BAL_COLS, budget=BUDGET)

print("Split (viabilidad):")
print(R.format_split(rep_honest))

In [ ]:
def pr(rep, reg):  return rep[reg]["pr_auc"]
def rc(rep, reg):  return rep[reg]["operating_point"]["recall"]
def rv(rep, reg):  return rep[reg]["operating_point"]["review_rate"]
def pv(rep, reg):  return rep[reg]["test_prevalence"]

print(f"Punto de operación: revisar ≤ {BUDGET:.0%} de operaciones.\n")
print("| Configuración | Recall@1% | Review rate | PR-AUC | prev. test |")
print("|---|---|---|---|---|")
print(f"| Balance + ALEATORIO (ingenuo, fuga×2)    | {rc(rep_leaky,'random'):.4f} | {rv(rep_leaky,'random'):.4f} | {pr(rep_leaky,'random'):.4f} | {pv(rep_leaky,'random'):.4f} |")
print(f"| Balance + temporal                       | {rc(rep_leaky,'temporal'):.4f} | {rv(rep_leaky,'temporal'):.4f} | {pr(rep_leaky,'temporal'):.4f} | {pv(rep_leaky,'temporal'):.4f} |")
print(f"| Honesto + ALEATORIO (optimista)          | {rc(rep_honest,'random'):.4f} | {rv(rep_honest,'random'):.4f} | {pr(rep_honest,'random'):.4f} | {pv(rep_honest,'random'):.4f} |")
print(f"| **Honesto + temporal (DESPLEGABLE)**     | {rc(rep_honest,'temporal'):.4f} | {rv(rep_honest,'temporal'):.4f} | {pr(rep_honest,'temporal'):.4f} | {pv(rep_honest,'temporal'):.4f} |")

# Eje A — fuga de ETIQUETA: medir con PR-AUC (RANKING). El balance rankea casi
# perfecto vs las features honestas. Comparable: mismo split, misma prevalencia.
gap_label_random   = pr(rep_leaky,'random')   - pr(rep_honest,'random')
gap_label_temporal = pr(rep_leaky,'temporal') - pr(rep_honest,'temporal')

# Eje B — fuga TEMPORAL: medir con RECALL@budget (punto de operación de negocio).
# El PR-AUC NO sirve aquí: los test temporal y aleatorio tienen distinta
# prevalencia, y el PR-AUC depende de la prevalencia.
gap_time_recall = rc(rep_honest,'random') - rc(rep_honest,'temporal')

print(f"\nEje A · fuga de ETIQUETA (PR-AUC, ranking):  random {gap_label_random:+.4f} · temporal {gap_label_temporal:+.4f}")
print(f"Eje B · fuga TEMPORAL (Recall@1%):           {gap_time_recall:+.4f}  (aleatorio sobreestima el fraude atrapado)")
print(f"\nFantasía -> realidad: recall {rc(rep_leaky,'random'):.1%} (balance+aleatorio) -> {rc(rep_honest,'temporal'):.1%} (honesto+temporal)")

## 7. Curvas PR — honesto temporal vs aleatorio

In [ ]:
import matplotlib.pyplot as plt
from evaluation.metrics import pr_curve

fig, ax = plt.subplots(figsize=(7,5))
for regime, color in [("random","tab:orange"),("temporal","tab:blue")]:
    y_te, sc = rep_honest["_scores"][regime]
    p, r, _ = pr_curve(y_te, sc)
    ax.plot(r, p, color=color, label=f"Honesto · {regime} (PR-AUC={rep_honest[regime]['pr_auc']:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Curva precision-recall — el optimismo del split aleatorio")
ax.legend(); plt.tight_layout(); plt.show()

## 8. Guardar reporte a GCS

In [ ]:
import json
from google.cloud import storage

rep_honest.pop("_scores", None)   # no serializable
out = {
    "budget": BUDGET,
    "honest": rep_honest,
    "leaky_balance": rep_leaky,
    "axis_A_label_leak_pr_auc": {"random": gap_label_random, "temporal": gap_label_temporal},
    "axis_B_temporal_leak_recall": gap_time_recall,
    "headline_recall_fantasy_to_deployable": {
        "fantasy_balance_random": rc(rep_leaky, "random"),
        "deployable_honest_temporal": rc(rep_honest, "temporal"),
    },
}
client = storage.Client(project=PROJECT_ID)
blob = client.bucket(BUCKET_FEAT).blob("feat-v1/evaluation/report.json")
blob.upload_from_string(json.dumps(out, indent=2, default=float), content_type="application/json")
print("Reporte escrito: gs://dupin-dupin-features/feat-v1/evaluation/report.json")

---
**Fase 3 completa** cuando este notebook corre limpio. El protagonista no es el
PR-AUC absoluto, sino el **gap descompuesto**: cuánto del rendimiento aparente era
fuga de etiqueta (eje A) y cuánto fuga temporal (eje B). El número honesto +
temporal es el único desplegable y el que entra a la model card (Fase 7).

La Fase 4 cambia el baseline (HistGradientBoosting) por el modelo final
(XGBoost/LightGBM), lo pasa por ESTE mismo régimen y publica el bundle.